# Этап 2. Анализ данных

Цель анализа - предоставить исчерпывающий анализ данных, на основе которого можно принять "правильное" бизнес-решение в сфере искусства и развлечений. Анализ призван ответить на следующий вопрос: "в какой фильм/сериал стоит вложить свои средства".

Как определить "успешный" проект? Наша команда выделяет три метрики "усмешности" проекта:

- Финансовый успех - показатель ROI
- Творческий успех - взвешенный рейтинг фильма
- Вовлеченность - число просмотренных минут зрителями, оценившими проект

Проекты с высоким ROI приносят студии финансовую прибыль, без которой ее деятельность невозможна. С другой стороны, успех студии — это не только кассовые сборы. Высокорейтинговые фильмы повышают статус студии, привлекают известных режиссёров и актеров, помогают получать инвестиции и определяют успех будущих проектов. Стриминг-сервисам могут быть полезны проекты с высокой вовлеченностью, поскольку для них важны просмотры, обсуждаемость и удержание аудитории.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('netflix_normalized_3.csv')

Перед началом анализа создадим необходимые ключевые признаки:

- Признак ROI (Финансовый успех) Показывает рентабельность: (Доходы - Расходы) / Расходы
- Признак weighted_rating (Творческий успех - формула Байеса из IMDB)
- Признак engagement (Вовлеченность)


In [ ]:
df['ROI'] = (df['revenue'] - df['budget']) / df['budget']

v = df['imdb_votecount']
R = df['vote_average_imdb']
C = R.mean()
m = v.quantile(0.50)  # Порог количества голосов (медиана)
df['weighted_rating'] = (v / (v + m) * R) + (m / (v + m) * C)

df['engagement'] = np.log1p(df['runtime'] * df['imdb_votecount'])

In [ ]:
df.head(3)

## Этап 2.1. "Идеальный" синтетический проект

Задача этапа: вывести синтетический "идеальный" проект с точки зрения финансов, рейтига и вовлеченности без учета скрытых взаимосвязей.


Перед началом анализа создадим все необходимые для этого этапа признаки:

- Признак is_movie\
  Если длительность больше 60 минут, то это скорее фильм (1), иначе — сериал (0).
- Преобразуем признаки под нужды анализа (разбиваем на квартили) - `runtime_group` и `budget_group`
- Создадим единую категорию главного жанра для группировки


In [ ]:
df['is_movie'] = (df['runtime'] >= 60).astype(int)

df['runtime_group'] = pd.qcut(
    df['runtime'], q=4, labels=['Short', 'Medium', 'Long', 'Very Long']
)
df['budget_group'] = pd.qcut(
    df['budget'], q=4, labels=['Low', 'Medium-Low', 'Medium-High', 'High']
)


def get_main_genre(row):
    for g in [
        'is_action',
        'is_comedy',
        'is_drama',
        'is_family',
        'is_thriller',
        'is_other',
    ]:
        if row[g]:
            return g.replace('is_', '')
    return 'unknown'


df['main_genre'] = df.apply(get_main_genre, axis=1)

In [ ]:
df.head(3)

### Этап 2.1.1. Синтетическая формула финансово-успешного фильма


Анализ для метрики ROI\
Сортируем по медиане и выводим ТОП-3


In [ ]:
stats = (
    df.groupby("runtime_group")["ROI"].agg(['median', 'var', 'mean', 'count']).dropna()
)
stats.sort_values(by='median', ascending=False).head(3)

In [ ]:
stats = (
    df.groupby("budget_group")["ROI"].agg(['median', 'var', 'mean', 'count']).dropna()
)
stats.sort_values(by='median', ascending=False).head(3)

In [ ]:
stats = df.groupby("is_movie")["ROI"].agg(['median', 'var', 'mean', 'count']).dropna()
stats.sort_values(by='median', ascending=False).head(3)

In [ ]:
stats = df.groupby("main_genre")["ROI"].agg(['median', 'var', 'mean', 'count']).dropna()
stats.sort_values(by='median', ascending=False).head(3)

In [ ]:
stats = df.groupby("age_group")["ROI"].agg(['median', 'var', 'mean', 'count']).dropna()
stats.sort_values(by='median', ascending=False).head(3)

### Этап 2.1.2. Синтетическая формула творчески-успешного фильма


Анализ для метрики weighted_rating\
Сортируем по медиане и выводим ТОП-3


In [ ]:
stats = (
    df.groupby("runtime_group")["weighted_rating"]
    .agg(['median', 'var', 'mean', 'count'])
    .dropna()
)
stats.sort_values(by='median', ascending=False).head(3)

In [ ]:
stats = (
    df.groupby("budget_group")["weighted_rating"]
    .agg(['median', 'var', 'mean', 'count'])
    .dropna()
)
stats.sort_values(by='median', ascending=False).head(3)

In [ ]:
stats = (
    df.groupby("is_movie")["weighted_rating"]
    .agg(['median', 'var', 'mean', 'count'])
    .dropna()
)
stats.sort_values(by='median', ascending=False).head(3)

In [ ]:
stats = (
    df.groupby("main_genre")["weighted_rating"]
    .agg(['median', 'var', 'mean', 'count'])
    .dropna()
)
stats.sort_values(by='median', ascending=False).head(3)

In [ ]:
stats = (
    df.groupby("age_group")["weighted_rating"]
    .agg(['median', 'var', 'mean', 'count'])
    .dropna()
)
stats.sort_values(by='median', ascending=False).head(3)

### Этап 2.1.3. Синтетическая формула фильма, вызвавшего наибольший интерес у зрителей


Анализ для метрики engagement\
Сортируем по медиане и выводим ТОП-3


In [ ]:
stats = (
    df.groupby("runtime_group")["engagement"]
    .agg(['median', 'var', 'mean', 'count'])
    .dropna()
)
stats.sort_values(by='median', ascending=False).head(3)

In [ ]:
stats = (
    df.groupby("budget_group")["engagement"]
    .agg(['median', 'var', 'mean', 'count'])
    .dropna()
)
stats.sort_values(by='median', ascending=False).head(3)

In [ ]:
stats = (
    df.groupby("is_movie")["engagement"]
    .agg(['median', 'var', 'mean', 'count'])
    .dropna()
)
stats.sort_values(by='median', ascending=False).head(3)

In [ ]:
stats = (
    df.groupby("main_genre")["engagement"]
    .agg(['median', 'var', 'mean', 'count'])
    .dropna()
)
stats.sort_values(by='median', ascending=False).head(3)

In [ ]:
stats = (
    df.groupby("age_group")["engagement"]
    .agg(['median', 'var', 'mean', 'count'])
    .dropna()
)
stats.sort_values(by='median', ascending=False).head(3)

### Этап 2.1.4. Промежуточные выводы


Таким образом, синтетические идеальные проекты с точки зрения финансов, рейтинга и зрительского интереса представлены ниже:


<table style="width:70%;"> <caption>Формула проекта с высоким ROI (Финансовый успех)</caption> <thead> <th>Характеристика</th> <th>Значение</th> </thead> <tbody> <tr> <td>Жанр</td> <td>Экшен (Action) или Комедия (Comedy)</td> </tr> <tr> <td>Длительность</td> <td>Очень длинные (Very Long) — от 110 минут и выше</td> </tr> <tr> <td>Бюджет</td> <td>От среднего к высокому (Medium-High)</td> </tr> <tr> <td>Возрастная аудитория</td> <td>Подростки (Teens)</td> </tr> <tr> <td>Сериал / Фильм</td> <td>Фильм</td> </tr> </tbody> </table>

_Справка: Группа "Short" и "Сериал" для ROI показала высокую медиану, но размер выборки (всего 3 проекта) делает этот результат статистически нерелевантным. Поэтому бесспорным уверенным лидером по окупаемости являются полнометражные фильмы._


<table style="width:70%;"> <caption>Формула проекта с высоким взвешенным рейтингом (Творческий успех)</caption> <thead> <th>Характеристика</th> <th>Значение</th> </thead> <tbody> <tr> <td>Жанр</td> <td>Драма (Drama)</td> </tr> <tr> <td>Длительность</td> <td>Средние (Medium)</td> </tr> <tr> <td>Бюджет</td> <td>Высокий (High)</td> </tr> <tr> <td>Возрастная аудитория</td> <td>Взрослые (Adults)</td> </tr> <tr> <td>Сериал / Фильм</td> <td>Сериал</td> </tr> </tbody> </table>


<table style="width:70%;"> <caption>Формула проекта с высокой вовлеченностью (Зрительский интерес)</caption> <thead> <th>Характеристика</th> <th>Значение</th> </thead> <tbody> <tr> <td>Жанр</td> <td>Драма (Drama)</td> </tr> <tr> <td>Длительность</td> <td>Очень длинные (Very Long)</td> </tr> <tr> <td>Бюджет</td> <td>Высокий (High)</td> </tr> <tr> <td>Возрастная аудитория</td> <td>Подростки (Teens)</td> </tr> <tr> <td>Сериал / Фильм</td> <td>Фильм</td> </tr> </tbody> </table>


Попробуем на основе этих данных синтетически определить портрет проекта, у которого все эти метрики будут высокими:

Анализ показывает, что универсального идеального рецепта не существует - параметры финансовой окупаемости (ROI) и признания критиков/зрителей (Рейтинг) требуют разных подходов. Тем не менее, "Драма" и "Экшен" показывают отличные результаты по вовлеченности и финансам, а целевая аудитория подростков (Teens) — самая выгодная как для сборов, так и для удержания внимания. Чтобы проект собрал кассу, удерживал внимание и получил приемлемые оценки, студии стоит вложиться в дорогой и длительный полнометражный фильм.

<table style="width:70%;"> <caption>Формула синтетического компромиссного «идеального» фильма</caption> <thead> <th>Характеристика</th> <th>Значение</th> </thead> <tbody> <tr> <td>Жанр</td> <td>Драма с элементами Экшена</td> </tr> <tr> <td>Длительность</td> <td>Очень длинный фильм (Very Long)</td> </tr> <tr> <td>Бюджет</td> <td>Высокий (High)</td> </tr> <tr> <td>Возрастная аудитория</td> <td>Подростки (Teens / PG-13 / TV-14)</td> </tr> <tr> <td>Сериал / Фильм</td> <td>Фильм</td> </tr> </tbody> </table>


## Этап 2.2. Корреляционный анализ


Задача этапа: провести _корреляционный анализ_ и определить, что влияет на выбранные метрики успешности проекта, и как они взаимосвязаны

В качестве метрики используется коэффициент Спирмена.


In [ ]:
cols_to_corr = [
    'budget',
    'runtime',
    'is_movie',
    'is_action',
    'is_comedy',
    'is_drama',
    'is_family',
    'is_thriller',
    'is_other',
]
corr_df = df[['ROI', 'weighted_rating', 'engagement'] + cols_to_corr].copy()
corr_df = pd.concat([corr_df, pd.get_dummies(df['age_group'], prefix='age')], axis=1)

spearman_corr = corr_df.corr(method='spearman')

### Этап 2.2.1. Корреляционный анализ ROI

Влияние признаков на ROI:


In [ ]:
spearman_corr[['ROI']].sort_values(by='ROI', ascending=False)

<table style="width:70%;"> <caption>Влияние признаков на ROI</caption> <thead> <tr> <th>Признак</th> <th>Влияние на ROI</th> </tr> </thead> <tbody> <tr> <td>Жанр</td> <td>Очень слабое: положительно у Action (+0.17) и Comedy (+0.12), отрицательно у Family (-0.19)</td> </tr> <tr> <td>Длительность</td> <td>Практически отсутствует (+0.04)</td> </tr> <tr> <td>Бюджет</td> <td>Практически отсутствует (+0.04)</td> </tr> <tr> <td>Возрастная аудитория</td> <td>Очень слабое: лучше у Adults (+0.09) и Teens (+0.05), хуже у Kids (-0.13)</td> </tr> <tr> <td>Сериал / Фильм</td> <td>Очень слабая отрицательная связь (-0.10)</td> </tr> </tbody> </table>


По результатам корреляционного анализа видно, что финансовая успешность проекта (ROI) в наибольшей степени связана с вовлеченностью аудитории: коэффициент Спирмена между ROI и engagement составляет 0.46, что соответствует слабой положительной корреляции, близкой к умеренной. Это означает, что проекты, вызывающие высокий интерес у зрителей, как правило, имеют и более высокую окупаемость.


### Этап 2.2.2. Корреляционный анализ рейтинга фильма

Влияние признаков на Творческий рейтинг (weighted_rating):


In [ ]:
spearman_corr[['weighted_rating']].sort_values(by='weighted_rating', ascending=False)

<table style="width:70%;"> <caption>Влияние признаков на weighted_rating</caption> <thead> <tr> <th>Признак</th> <th>Влияние на weighted_rating</th> </tr> </thead> <tbody> <tr> <td>Жанр</td> <td>Слабое положительное у Drama (+0.26) и Thriller (+0.25), слабое отрицательное у Family (-0.22), Comedy (-0.15), Action (-0.11)</td> </tr> <tr> <td>Длительность</td> <td>Слабая отрицательная связь (-0.41)</td> </tr> <tr> <td>Бюджет</td> <td>Практически отсутствует (+0.02)</td> </tr> <tr> <td>Возрастная аудитория</td> <td>Слабое положительное у Adults (+0.14), отрицательное у Kids (-0.21)</td> </tr> <tr> <td>Сериал / Фильм</td> <td>Слабая отрицательная связь (-0.40), то есть более высокие оценки чаще характерны для сериалов</td> </tr> </tbody> </table>


Для метрики weighted_rating наиболее сильная связь обнаружена с длительностью проекта (runtime = -0.41) и форматом фильм / сериал (is_movie = -0.40). Это уже слабая отрицательная корреляция, близкая к умеренной. Следовательно, более короткие проекты и, вероятно, сериалы в среднем получают более высокие оценки, чем длинные полнометражные фильмы.


### Этап 2.2.3. Корреляционный анализ вовлеченности

Влияние признаков на Вовлеченность (engagement):


In [ ]:
spearman_corr[['engagement']].sort_values(by='engagement', ascending=False)

<table style="width:70%;"> <caption>Влияние признаков на engagement</caption> <thead> <tr> <th>Признак</th> <th>Влияние на engagement</th> </tr> </thead> <tbody> <tr> <td>Жанр</td> <td>Слабое положительное у Thriller (+0.30) и Drama (+0.28), отрицательное у Family (-0.28)</td> </tr> <tr> <td>Длительность</td> <td>Слабая положительная связь (+0.39)</td> </tr> <tr> <td>Бюджет</td> <td>Умеренная положительная связь (+0.56)</td> </tr> <tr> <td>Возрастная аудитория</td> <td>Слабая положительная у Teens (+0.33), отрицательная у Kids (-0.46)</td> </tr> <tr> <td>Сериал / Фильм</td> <td>Слабая положительная связь (+0.25), вовлеченность выше у фильмов</td> </tr> </tbody> </table>


Метрика engagement демонстрирует наиболее выраженные зависимости среди всех трех показателей успеха.

Наиболее сильная положительная связь наблюдается между вовлеченностью и бюджетом:
budget = 0.56, что соответствует умеренной положительной корреляции.
Это означает, что дорогие проекты в среднем вызывают больший интерес аудитории и собирают больше просмотренных минут.

Вторая важная связь - между engagement и ROI (0.46). Это подтверждает, что высокий интерес аудитории в значительной степени помогает проекту быть коммерчески успешным.


### Этап 2.2.4. Влияние метрик успешности друг на друга


In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(
    spearman_corr.loc[
        ['ROI', 'weighted_rating', 'engagement'],
        ['ROI', 'weighted_rating', 'engagement'],
    ],
    annot=True,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    fmt=".2f",
)
plt.title("Корреляция Спирмена между тремя метриками")
plt.show()

<table style="width:70%;"> <caption>Корреляция между ROI, weighted_rating и engagement</caption> <thead> <tr> <th>Пара метрик</th> <th>Характер связи</th> </tr> </thead> <tbody> <tr> <td>ROI - weighted_rating</td> <td>Слабая отрицательная (-0.14)</td> </tr> <tr> <td>ROI - engagement</td> <td>Слабая положительная, близкая к умеренной (+0.46)</td> </tr> <tr> <td>weighted_rating - engagement</td> <td>Слабая положительная (+0.19)</td> </tr> </tbody> </table>


По тепловой карте корреляций между тремя метриками успешности можно сделать следующие выводы:

- Между ROI и weighted_rating наблюдается слабая отрицательная корреляция: -0.14.
  Это означает, что финансово успешный фильм не обязательно получает высокие зрительские оценки.

- Между ROI и engagement наблюдается слабая положительная корреляция, близкая к умеренной: 0.46.
  Следовательно, интерес аудитории действительно помогает коммерческому успеху, хотя и не определяет его полностью.

- Между weighted_rating и engagement наблюдается слабая положительная корреляция: 0.19.
  Это означает, что более высоко оцененные проекты в среднем вызывают больший зрительский интерес, однако связь здесь заметно слабее, чем можно было ожидать.

Делая итоговый выдо: можно сказать следующее:
Финансовый, творческий и зрительский успех - это три разные стороны успешности проекта, которые пересекаются лишь частично.
Наиболее тесно между собой связаны ROI и engagement, то есть коммерческий успех в большей степени определяется интересом аудитории, чем высоким рейтингом. В то же время высокий рейтинг не гарантирует ни кассового успеха, ни рекордной вовлеченности.

Следовательно, при принятии бизнес-решения нельзя ориентироваться только на одну метрику.
Если студия хочет заработать, ей нужен проект с высоким потенциалом вовлечения.
Если цель - престиж и репутация, лучше делать ставку на драматический или триллерный контент.
Если приоритет - удержание зрителя на платформе, наиболее перспективны дорогие, продолжительные проекты, ориентированные на подростковую аудиторию.
